# Accessing Copernicus Marine data with the Copernicus Marine Toolbox

This notebook demonstrates how to download a subset of Copernicus Marine data using the current **Copernicus Marine Toolbox** Python API.

The Copernicus Marine Toolbox provides the current programmatic interface for exploring catalogue metadata, subsetting datasets, downloading original files, and accessing datasets remotely.

Useful resources:
- [Copernicus Marine Data Store](https://data.marine.copernicus.eu/products)
- [Copernicus Marine Toolbox – Introduction](https://help.marine.copernicus.eu/en/articles/7949409-copernicus-marine-toolbox-introduction)
- [Copernicus Marine Toolbox – Installation](https://help.marine.copernicus.eu/en/articles/7970514-copernicus-marine-toolbox-installation)


## 1. Copernicus Marine Toolbox

The Copernicus Marine Toolbox is available in the EOxHub kernel environment and provides the Python interface used in this notebook to access Copernicus Marine data.

The installed version can be checked with:

In [ ]:
import copernicusmarine

print(f"Copernicus Marine Toolbox version: {copernicusmarine.__version__}")


## 2. Authentication

The Copernicus Marine Toolbox requires a Copernicus Marine account for data access. Credentials should **not** be written directly into the notebook.

For automated workflows, the Toolbox supports the environment variables `COPERNICUSMARINE_SERVICE_USERNAME` and `COPERNICUSMARINE_SERVICE_PASSWORD`. These can be configured by the EOxHub Workspaces or by the user.

If credentials are already configured in the EOxHub Workspaces, the following cells can be used to check whether the expected variables are available without displaying their values.

In [ ]:
import os

username_configured = bool(os.environ.get("COPERNICUSMARINE_SERVICE_USERNAME"))
password_configured = bool(os.environ.get("COPERNICUSMARINE_SERVICE_PASSWORD"))

print(f"Username configured: {username_configured}")
print(f"Password configured: {password_configured}")


If the credentials are not available as environment variables, they can also be configured with the Toolbox's `login` functionality. See the official [credentials configuration documentation](https://help.marine.copernicus.eu/en/articles/7949409-copernicus-marine-toolbox-introduction) for the available methods.

## 3. Explore the Copernicus Marine catalogue

The Toolbox provides the `describe()` function to search the catalogue and retrieve metadata for products, datasets and variables. This is useful because dataset IDs can change as products evolve, allowing the current dataset to be identified directly from the catalogue.

In [ ]:
catalogue = copernicusmarine.describe(
    contains=["GLOBAL_ANALYSISFORECAST_PHY_001_024"]
)

print(type(catalogue))


The current Copernicus Marine product catalogue contains the **Global Ocean Physics Analysis and Forecast** product under `GLOBAL_ANALYSISFORECAST_PHY_001_024`. The product currently provides hourly, daily and monthly datasets. The product page should be consulted when selecting the dataset and variable needed for a particular request.

In [ ]:
# Display a compact representation of the catalogue response.
# The exact structure can change as the catalogue evolves.
catalogue.model_dump().keys()


## 4. Select the dataset and subset

The request retrieves the eastward ocean current (`uo`) for a region around the Black Sea, for a specified time period and near-surface depth.

The request is performed using the Copernicus Marine Toolbox `subset()` API.

The current product page lists the hourly dataset as `cmems_mod_glo_phy_anfc_0.083deg_PT1H-m`. This dataset provides the ocean current data used for the request.


In [ ]:
dataset_id = "cmems_mod_glo_phy_anfc_0.083deg_PT1H-m"
variable = "uo"

longitude_min = 28.5
longitude_max = 32.5
latitude_min = 42.5
latitude_max = 46.0

start_datetime = "2022-06-01 00:00:00"
end_datetime = "2022-06-01 01:00:00"

minimum_depth = 0.494024
maximum_depth = 0.494026


**Note on the dates:** The current hourly dataset listed on the product page starts on 1 June 2022, so the example dates use a period currently covered by the dataset. Check the product page and `describe()` output when selecting a different time period.


## 5. Download the subset

The `subset()` function downloads only the requested part of the dataset. For gridded datasets, NetCDF is the default output format.

The variable, spatial extent, time range and depth range are specified directly as Python arguments, allowing the required subset of the dataset to be downloaded efficiently.

The `skip_existing=True` option prevents the Toolbox from downloading the same subset again when the output file already exists.

In [ ]:
import getpass

username = getpass.getpass("Copernicus Marine username: ")
password = getpass.getpass("Copernicus Marine password: ")

output_directory = "."

response = copernicusmarine.subset(
    dataset_id=dataset_id,
    variables=[variable],
    start_datetime=start_datetime,
    end_datetime=end_datetime,
    minimum_longitude=longitude_min,
    maximum_longitude=longitude_max,
    minimum_latitude=latitude_min,
    maximum_latitude=latitude_max,
    minimum_depth=minimum_depth,
    maximum_depth=maximum_depth,
    output_directory=output_directory,
    skip_existing=True,
    username=username,
    password=password,
)

response

## 6. Open the downloaded NetCDF file

The returned response contains information about the generated output. We can use `xarray` to inspect the downloaded NetCDF dataset.

In [ ]:
import xarray as xr

output_file = response[0].file_path if isinstance(response, list) else response.file_path
print(output_file)

ds = xr.open_dataset(output_file)
ds

## 7. Inspect the downloaded variable


In [ ]:
ds[variable]


In [ ]:
import matplotlib.pyplot as plt

ds[variable].isel(time=0, depth=0).plot.imshow(
    figsize=(12, 8),
    cmap="viridis"
)

plt.title("Eastward ocean current velocity (uo)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

## Summary

The Copernicus Marine Toolbox provides the tools used to access and process Copernicus Marine data:

* `copernicusmarine.describe()` can be used to explore the current catalogue and identify datasets and variables.
* `copernicusmarine.subset()` downloads only the requested spatial, temporal, depth and variable subset.
* The downloaded NetCDF data can then be processed with standard Python tools such as `xarray`.

This workflow follows the current Copernicus Marine Toolbox documentation and provides a direct way to access and work with the required ocean data.
